# Sideris Fine-Tuning Pipeline v3 (Stage 0 CPT + SFT + DPO + GRPO | Kaggle T4 x2 / Colab T4)

Train **Sideris Aether** (`Qwen3.5-4B`), **Sideris Lux** (`Qwen3.5-2B`), **Sideris Orion** (`Phi-4-mini-reasoning`), or **Sideris Titan** (`Qwen3.5-9B`) from scratch or from a saved Hugging Face LoRA checkpoint (`RESUME_FROM_ADAPTER`).

### Before clicking Save & Run All (Commit) in Kaggle
1. **Session options -> Accelerator:** `GPU T4 x2` (Cell 1 locks `CUDA_VISIBLE_DEVICES=0`, single T4 15 GB)
2. **Session options -> Internet:** `On`
3. **Add-ons -> Secrets:** enable `HF_TOKEN` (Hugging Face `Write` token for private `SiderisAI` org)
4. **Input -> + Add Input:**
   - **`Sideris-Data`** *(required, Private)*: `train_sft_enriched.jsonl`, `train_dpo_enriched.jsonl`, `train_grpo.jsonl`, `eval_holdout.jsonl`, `rewards.py`
   - **`Sideris-Books`** *(optional, Private)*: `books_cpt.jsonl` (or `books_cpt.zip`); Stage 0 CPT skips cleanly if not attached

### Default preset: Aether v2.1 (one session, ~9.7 h)
- Resumes `SiderisAI/Sideris-Aether-sft`, finishes SFT, short DPO, longer GRPO, eval vs base, export to `SiderisAI/Sideris-Aether-v2.1*`.

### Recommended Two-Session Schedule for Sideris Aether (Qwen3.5-4B, 12 h Kaggle limit)
- **Session 1 (default below):** Stage 0 CPT (`~3.0-3.6 h`) + Stage 1 SFT 2 epochs (`~5.5-6.8 h`) + Holdout Eval -> automatically pushes private checkpoints `SiderisAI/Sideris-Aether-cpt` and `SiderisAI/Sideris-Aether-sft`.
- **Session 2:** Set `RESUME_FROM_ADAPTER = "SiderisAI/Sideris-Aether-sft"`, `RUN_CPT = False`, `RUN_SFT = False`, `RUN_DPO = True`, `RUN_GRPO = True`, `EVAL_COMPARE_BASE = True`, `RUN_EXPORT = True` -> runs DPO + GRPO + Base-vs-Tuned Holdout Eval + 16-bit Merge + GGUF Q4_K_M export (`~4.5-6.0 h`).

In [ ]:
# 1. Environment Guard, Configuration, Auto-Discovery & HF/Telemetry Helpers
import json, os, pathlib, shutil, subprocess, time, zipfile

os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Single T4 prevents DataParallel crashes with Unsloth & 4-bit
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---- User Configuration (preset: Aether v2.1, one session inside an 11.5 h weekly quota) ----
MODEL_CHOICE          = "aether"       # "lux" | "aether" | "orion" | "titan"
HF_USERNAME           = "SiderisAI"    # Private HF org for automatic checkpoint & final uploads
RUN_TAG               = "v2.1"         # Suffix of every HF repo (Sideris-Aether-v2.1-*), so v2 is never overwritten
RESUME_FROM_ADAPTER   = "SiderisAI/Sideris-Aether-sft"  # v2 SFT (CPT + 438 SFT steps); "" = from base model
SESSION_MAX_HOURS     = 11.0           # Wall-clock cap (setup + pip run before this clock starts)

# Stage 0 — CPT on reference textbooks (done in v2; enable for a fresh run with the 19-book corpus)
RUN_CPT               = False
CPT_EPOCHS            = 1
CPT_MAX_STEPS         = 95             # ~3.4 min/step on T4 at 4096 tokens
CPT_LR                = 5e-5
CPT_MAX_SEQ_LEN       = 4096
CPT_FALLBACK_SEQ_LEN  = 2048
CPT_SFT_REPLAY_RATIO  = 0.12
CPT_SUBJECT_WEIGHTS   = {
    "fluid-mechanics": 1.0,
    "aerospace-materials-1": 0.65,
    "engineering-mechanics": 1.35,
    "advanced-maths": 1.40,
    "business-management": 1.0,
}

# Stage 1 — SFT continuation: v2 stopped at 438/742 steps with the cosine LR not annealed
RUN_SFT               = True
SFT_EPOCHS            = 1
SFT_MAX_STEPS         = 330            # ~46 s/step on T4 -> ~4.2 h (~0.9 epoch more; -1 = full epochs)
SFT_LR                = 1e-4           # Half the recipe LR: continuing an adapter, cosine anneals to 0
MAX_SEQ_LEN           = 2048

# Stage 2 — DPO: v2 loss hit 0.0003 (synthetic pairs too easy) -> fewer steps, softer update
RUN_DPO               = True
DPO_MAX_STEPS         = 60             # ~92 s/step on T4 -> ~1.5 h (v2 used 160 steps / 4.1 h)
DPO_LR                = 2e-6
DPO_BETA              = 0.2

# Stage 3 — GRPO: longer run + extra reward for a final \boxed{} answer on problems
RUN_GRPO              = True
GRPO_MAX_STEPS        = 120            # ~1 min/step on T4 -> ~2.0 h
GRPO_BOXED_REWARD     = True

# Evaluation & Export
RUN_EVAL              = True
EVAL_MAX_SAMPLES      = 60             # The first 30 are the same prompts as the v2 evaluation
EVAL_MAX_NEW_TOKENS   = 768
EVAL_COMPARE_BASE     = True           # ~1.45 min/sample with base; set False to halve the eval time
RUN_EXPORT            = True           # LoRA + merged 16-bit + GGUF Q4_K_M to private SiderisAI repos

# Planned budget (T4): setup ~15 min + SFT ~253 + DPO ~93 + GRPO ~120 + eval ~87 + export ~12 = ~9.7 h.
# Each stage stops early if needed so that the later stages still fit (PLANNED_MIN_AFTER_*).

# --- Session 1 Preset (fresh run from base: CPT + SFT) ---
# RUN_TAG = "v3"; RESUME_FROM_ADAPTER = ""
# RUN_CPT = True; RUN_SFT = True; SFT_EPOCHS = 2; SFT_MAX_STEPS = -1; SFT_LR = 2e-4
# RUN_DPO = False; RUN_GRPO = False; EVAL_COMPARE_BASE = False; RUN_EXPORT = False

# --- Single-Session Express Preset (< 10.5 h without CPT, or < 10.8 h with SFT_EPOCHS=1 + CPT_MAX_STEPS=45) ---
# CPT_MAX_STEPS = 45; SFT_EPOCHS = 1; RUN_DPO = True; RUN_GRPO = True; GRPO_MAX_STEPS = 50; RUN_EXPORT = True

RECIPES = {
    "lux":    {"name": "Sideris-Lux",    "base": "Qwen/Qwen3.5-2B",                "fallback": "Qwen/Qwen3-1.7B", "r": 16, "lr": 2e-4},
    "aether": {"name": "Sideris-Aether", "base": "Qwen/Qwen3.5-4B",                "fallback": "Qwen/Qwen3-4B",   "r": 32, "lr": 2e-4},
    "orion":  {"name": "Sideris-Orion",  "base": "microsoft/Phi-4-mini-reasoning", "fallback": "Qwen/Qwen3-4B",   "r": 32, "lr": 1.5e-4},
    "titan":  {"name": "Sideris-Titan",  "base": "Qwen/Qwen3.5-9B",                "fallback": "Qwen/Qwen3-8B",   "r": 32, "lr": 1.5e-4},
}
RECIPE = RECIPES[MODEL_CHOICE]
HF_REPO_BASE = RECIPE["name"] + (f"-{RUN_TAG}" if RUN_TAG else "")   # e.g. Sideris-Aether-v2.1

# Planned minutes of every stage AFTER SFT / DPO (measured on Kaggle T4, Aether 4B), so an earlier
# stage stops in time to leave room for the later ones instead of eating the whole session.
MIN_PER_DPO_STEP, MIN_PER_GRPO_STEP, MIN_PER_EVAL_SAMPLE = 1.55, 1.0, (1.45 if EVAL_COMPARE_BASE else 0.75)
PLANNED_MIN_GRPO = GRPO_MAX_STEPS * MIN_PER_GRPO_STEP if RUN_GRPO else 0.0
PLANNED_MIN_EVAL_EXPORT = (EVAL_MAX_SAMPLES * MIN_PER_EVAL_SAMPLE if RUN_EVAL else 0.0) + (12.0 if RUN_EXPORT else 0.0)
PLANNED_MIN_AFTER_DPO = PLANNED_MIN_GRPO + PLANNED_MIN_EVAL_EXPORT
PLANNED_MIN_AFTER_SFT = (DPO_MAX_STEPS * MIN_PER_DPO_STEP if RUN_DPO else 0.0) + PLANNED_MIN_AFTER_DPO

SESSION_START = time.time()
SESSION_DEADLINE = SESSION_START + SESSION_MAX_HOURS * 3600.0
STAGE_METRICS = {}

def elapsed_min():
    return (time.time() - SESSION_START) / 60.0

def remaining_min():
    return max(0.0, (SESSION_DEADLINE - time.time()) / 60.0)

def get_hf_token():
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok.strip()
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
        return tok.strip() if tok else None
    except Exception:
        return None

# Verify GPU is actually attached
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError("No GPU found! Open Kaggle 'Session options' (bottom-right) -> Accelerator = 'GPU T4 x2' and Internet = 'On'.")
print("GPU ready:", gpu.stdout.strip().splitlines()[0])

# Auto-discover Sideris-Data anywhere under /kaggle/input or /content
DATA_DIR = None
for root in ("/kaggle/input", "/content", "."):
    p = pathlib.Path(root)
    if not p.exists():
        continue
    for target in ("train_sft_enriched.jsonl", "train_sft.jsonl"):
        hits = list(p.rglob(target))
        if hits:
            DATA_DIR = str(hits[0].parent)
            break
    if DATA_DIR:
        break

if not DATA_DIR:
    raise FileNotFoundError("Could not find train_sft_enriched.jsonl or train_sft.jsonl under /kaggle/input or /content. Attach Sideris-Data via '+ Add Input'.")

# Auto-discover private Sideris-Books (optional: skips Stage 0 CPT cleanly if missing)
BOOKS_FILE = None
for root in ("/kaggle/input", "/content", "tools/ai_training/out", "."):
    p = pathlib.Path(root)
    if not p.exists():
        continue
    hits = sorted(p.rglob("books_cpt.jsonl"))
    if hits:
        BOOKS_FILE = str(hits[0])
        break
    zips = sorted(p.rglob("books_cpt*.zip"))
    if zips:
        extract_dir = pathlib.Path("/tmp/sideris_books")
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zips[0], "r") as zf:
            members = [m for m in zf.namelist() if m.endswith("books_cpt.jsonl")]
            if members:
                target_out = extract_dir / "books_cpt.jsonl"
                with zf.open(members[0]) as src_f, open(target_out, "wb") as dst_f:
                    shutil.copyfileobj(src_f, dst_f)
                BOOKS_FILE = str(target_out)
                break

OUT_DIR = "/kaggle/working/sideris_out" if os.path.exists("/kaggle/working") else "/content/sideris_out"
os.makedirs(OUT_DIR, exist_ok=True)

def save_metrics():
    payload = {
        "model_choice": MODEL_CHOICE,
        "recipe": RECIPE,
        "resume_from_adapter": RESUME_FROM_ADAPTER,
        "elapsed_min": round(elapsed_min(), 2),
        "stages": STAGE_METRICS,
    }
    mpath = os.path.join(OUT_DIR, "training_metrics.json")
    with open(mpath, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    return mpath

def push_folder_to_hf(folder_path, repo_id, commit_message="Sideris checkpoint"):
    token = get_hf_token()
    if not token or not HF_USERNAME or HF_USERNAME == "YOUR_HF_USER":
        print(f"HF upload skipped for {repo_id}: missing HF_TOKEN or HF_USERNAME.")
        return None
    if not folder_path or not os.path.isdir(folder_path):
        print(f"HF upload skipped for {repo_id}: missing directory {folder_path}.")
        return None
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=token)
        api.create_repo(repo_id=repo_id, exist_ok=True, private=True)
        api.upload_folder(folder_path=folder_path, repo_id=repo_id, commit_message=commit_message)
        print(f"HF private upload complete -> https://huggingface.co/{repo_id}")
        return repo_id
    except Exception as e:
        print(f"HF upload warning for {repo_id}: {type(e).__name__}: {str(e)[:300]}")
        return None

def cleanup_trainer_checkpoints(stage_dir):
    if not stage_dir or not os.path.isdir(stage_dir):
        return
    for name in os.listdir(stage_dir):
        if name.startswith("checkpoint-"):
            shutil.rmtree(os.path.join(stage_dir, name), ignore_errors=True)

print("Recipe              :", RECIPE)
print("DATA_DIR            :", DATA_DIR)
print("BOOKS_FILE          :", BOOKS_FILE if BOOKS_FILE else "Not attached (Stage 0 CPT will be skipped)")
print("OUT_DIR             :", OUT_DIR)
print("RESUME_FROM_ADAPTER :", RESUME_FROM_ADAPTER or "(none - training from base model)")
print("HF_TOKEN detected   :", bool(get_hf_token()))
print(f"Session budget      : {SESSION_MAX_HOURS:.1f} h ({remaining_min():.0f} min)")
for fname in sorted(os.listdir(DATA_DIR)):
    fpath = os.path.join(DATA_DIR, fname)
    if os.path.isfile(fpath):
        print(f"  - {fname} ({os.path.getsize(fpath)/1024:.1f} KB)")

In [ ]:
# 2. Install Compatible Dependencies (Transformers >= 5.2.0 for Qwen3.5 + Unsloth, WITHOUT touching Kaggle's torch)
import subprocess, sys

def pip_run(*args):
    cmd = [sys.executable, "-m", "pip", *args]
    print(">>", " ".join(cmd))
    subprocess.check_call(cmd)

# NOTE: Do NOT install optimum[onnxruntime] here (it downgrades transformers to 4.57 and breaks Qwen3.5!)
# Also remove broken preinstalled xformers mismatch warning on torch 2.11+cu130 if present.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "xformers"], check=False)

pip_run("install", "-q", "--upgrade",
        "transformers>=5.2.0",
        "trl>=0.15.0",
        "peft>=0.15.0",
        "accelerate>=1.5.0",
        "bitsandbytes>=0.45.0",
        "datasets",
        "sympy",
        "sentencepiece",
        "protobuf",
        "huggingface_hub",
        "hf_transfer")

pip_run("install", "-q", "--upgrade", "--no-deps", "unsloth_zoo", "unsloth")

import torch, transformers, trl, peft, bitsandbytes
print(f"torch={torch.__version__} (cuda={torch.cuda.is_available()})")
print(f"transformers={transformers.__version__} | trl={trl.__version__} | peft={peft.__version__} | bnb={bitsandbytes.__version__}")
if tuple(int(x) for x in transformers.__version__.split(".")[:2]) < (5, 2):
    print("NOTICE: transformers was upgraded in this session; if Cell 3 reports an old cached transformers version, click 'Run -> Restart & clear cell outputs' once and re-run from Cell 1 (pip will be instant).")

In [ ]:
# 3. Load Base Model (4-bit QLoRA) or Resume Adapter + Shared Training Helpers
import gc, inspect, json, os, shutil

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Always import unsloth BEFORE transformers/trl/peft!
USE_UNSLOTH = False
try:
    import unsloth
    from unsloth import FastLanguageModel
    UNSLOTH_AVAILABLE = True
except Exception as e:
    UNSLOTH_AVAILABLE = False
    print("Unsloth import skipped, will use standard HF 4-bit QLoRA:", e)

import torch
from transformers import TrainerCallback

if MODEL_CHOICE == "orion":
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)

target_mods = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"] + (
    ["qkv_proj", "gate_up_proj"] if MODEL_CHOICE == "orion" else []
)

model = None
tokenizer = None
LOADED_BASE = None
active_adapter = None

def unwrap_tokenizer(tok):
    # Qwen3.5 VLM processors wrap the text tokenizer under `.tokenizer`
    return getattr(tok, "tokenizer", tok)

LOAD_SEQ_LEN = max(MAX_SEQ_LEN, CPT_MAX_SEQ_LEN if (RUN_CPT and BOOKS_FILE) else MAX_SEQ_LEN)

if RESUME_FROM_ADAPTER:
    hf_tok = get_hf_token()
    if os.path.isdir(RESUME_FROM_ADAPTER):
        local_resume_dir = RESUME_FROM_ADAPTER
    else:
        from huggingface_hub import snapshot_download
        local_resume_dir = os.path.join(OUT_DIR, "resumed-adapter")
        print(f"Downloading private adapter {RESUME_FROM_ADAPTER} -> {local_resume_dir} ...")
        snapshot_download(repo_id=RESUME_FROM_ADAPTER, token=hf_tok, local_dir=local_resume_dir)

    adapter_base = RECIPE["base"]
    cfg_json = os.path.join(local_resume_dir, "adapter_config.json")
    if os.path.isfile(cfg_json):
        with open(cfg_json, encoding="utf-8") as f:
            adapter_base = json.load(f).get("base_model_name_or_path") or adapter_base

    if UNSLOTH_AVAILABLE:
        try:
            fp_params = set(inspect.signature(FastLanguageModel.from_pretrained).parameters)
            fp_kwargs = dict(
                model_name=local_resume_dir,
                max_seq_length=LOAD_SEQ_LEN,
                dtype=None,
                load_in_4bit=True,
                token=hf_tok,
            )
            if "text_only" in fp_params and "qwen3.5" in str(adapter_base).lower():
                fp_kwargs["text_only"] = True
            m, t = FastLanguageModel.from_pretrained(**fp_kwargs)
            FastLanguageModel.for_training(m)
            model, tokenizer = m, unwrap_tokenizer(t)
            USE_UNSLOTH = True
            LOADED_BASE = adapter_base
            print(f"SUCCESS: Resumed adapter {local_resume_dir} directly via Unsloth!")
        except Exception as e_ru:
            print(f"  Direct Unsloth adapter load failed ({type(e_ru).__name__}: {str(e_ru)[:250]}), trying base + PeftModel ...")
            gc.collect()
            torch.cuda.empty_cache()
            try:
                from peft import PeftModel
                fp_params = set(inspect.signature(FastLanguageModel.from_pretrained).parameters)
                fp_kwargs = dict(
                    model_name=adapter_base,
                    max_seq_length=LOAD_SEQ_LEN,
                    dtype=None,
                    load_in_4bit=True,
                )
                if "text_only" in fp_params and "qwen3.5" in str(adapter_base).lower():
                    fp_kwargs["text_only"] = True
                m, t = FastLanguageModel.from_pretrained(**fp_kwargs)
                m = PeftModel.from_pretrained(m, local_resume_dir, is_trainable=True)
                try:
                    FastLanguageModel.for_training(m)
                except Exception:
                    pass
                if hasattr(m, "gradient_checkpointing_enable"):
                    m.gradient_checkpointing_enable()
                if hasattr(m, "enable_input_require_grads"):
                    m.enable_input_require_grads()
                model, tokenizer = m, unwrap_tokenizer(t)
                USE_UNSLOTH = True
                LOADED_BASE = adapter_base
                print(f"SUCCESS: Resumed adapter via Unsloth base + PeftModel!")
            except Exception as e_ru2:
                print(f"  Unsloth resume fallback failed ({type(e_ru2).__name__}: {str(e_ru2)[:250]})")
                gc.collect()
                torch.cuda.empty_cache()

    if model is None:
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        from peft import PeftModel, prepare_model_for_kbit_training

        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
        t = AutoTokenizer.from_pretrained(adapter_base, trust_remote_code=True)
        m = AutoModelForCausalLM.from_pretrained(
            adapter_base,
            quantization_config=bnb_cfg,
            device_map={"": 0},
            trust_remote_code=True,
        )
        m.config.use_cache = False
        m = prepare_model_for_kbit_training(m, use_gradient_checkpointing=True)
        m = PeftModel.from_pretrained(m, local_resume_dir, is_trainable=True)
        model, tokenizer = m, unwrap_tokenizer(t)
        USE_UNSLOTH = False
        LOADED_BASE = adapter_base
        print(f"SUCCESS: Resumed adapter via HuggingFace 4-bit NF4 + PeftModel!")

    for pname, p in model.named_parameters():
        if "lora_" in pname:
            p.requires_grad = True
    model.config.use_cache = False
    active_adapter = local_resume_dir
else:
    for candidate in (RECIPE["base"], RECIPE["fallback"]):
        print(f"\nLoading {candidate} from scratch (4-bit QLoRA) ...")
        if UNSLOTH_AVAILABLE:
            try:
                fp_params = set(inspect.signature(FastLanguageModel.from_pretrained).parameters)
                fp_kwargs = dict(
                    model_name=candidate,
                    max_seq_length=LOAD_SEQ_LEN,
                    dtype=None,
                    load_in_4bit=True,
                )
                if "text_only" in fp_params and "qwen3.5" in candidate.lower():
                    fp_kwargs["text_only"] = True
                m, t = FastLanguageModel.from_pretrained(**fp_kwargs)
                m = FastLanguageModel.get_peft_model(
                    m,
                    r=RECIPE["r"],
                    lora_alpha=RECIPE["r"],
                    target_modules=target_mods,
                    lora_dropout=0,
                    bias="none",
                    use_gradient_checkpointing="unsloth",
                    random_state=42,
                )
                model, tokenizer = m, unwrap_tokenizer(t)
                USE_UNSLOTH = True
                LOADED_BASE = candidate
                print(f"SUCCESS: Loaded {candidate} via Unsloth FastLanguageModel (4-bit QLoRA)!")
                break
            except Exception as e_u:
                print(f"  Unsloth failed for {candidate}: {type(e_u).__name__}: {str(e_u)[:300]}")
                gc.collect()
                torch.cuda.empty_cache()

        # Fallback: pure HuggingFace Transformers + 4-bit NF4 BitsAndBytes + PEFT LoRA
        try:
            from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
            from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

            bnb_cfg = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.float16,
            )
            t = AutoTokenizer.from_pretrained(candidate, trust_remote_code=True)
            m = AutoModelForCausalLM.from_pretrained(
                candidate,
                quantization_config=bnb_cfg,
                device_map={"": 0},
                trust_remote_code=True,
            )
            m.config.use_cache = False
            m = prepare_model_for_kbit_training(m, use_gradient_checkpointing=True)
            lora_cfg = LoraConfig(
                r=RECIPE["r"],
                lora_alpha=RECIPE["r"],
                target_modules=target_mods,
                lora_dropout=0.05,
                bias="none",
                task_type="CAUSAL_LM",
            )
            m = get_peft_model(m, lora_cfg)
            model, tokenizer = m, unwrap_tokenizer(t)
            USE_UNSLOTH = False
            LOADED_BASE = candidate
            print(f"SUCCESS: Loaded {candidate} via HuggingFace 4-bit NF4 + PEFT!")
            break
        except Exception as e_hf:
            print(f"  HF fallback failed for {candidate}: {type(e_hf).__name__}: {str(e_hf)[:300]}")
            gc.collect()
            torch.cuda.empty_cache()

if model is None:
    raise RuntimeError("Could not load primary, fallback, or resumed model. If transformers was just upgraded, click Session -> Restart once and re-run.")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def format_chat(messages, add_generation_prompt=False):
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=add_generation_prompt, enable_thinking=False
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=add_generation_prompt
        )

class TimeBudgetCallback(TrainerCallback):
    def __init__(self, reserve_min=40.0, label="stage"):
        self.reserve_min = reserve_min
        self.label = label
        self.stopped_early = False

    def _check(self, state, control):
        rem = remaining_min()
        if rem < self.reserve_min:
            print(f"[{self.label}] Time budget guard triggered at step {state.global_step} ({rem:.1f} min left < {self.reserve_min:.0f} min reserve). Stopping cleanly.")
            self.stopped_early = True
            control.should_training_stop = True
        return control

    def on_log(self, args, state, control, **kwargs):
        return self._check(state, control)

    def on_step_end(self, args, state, control, **kwargs):
        return self._check(state, control)

class EpochPushCallback(TrainerCallback):
    def __init__(self, mdl, tok, save_dir, hf_repo):
        self.mdl = mdl
        self.tok = tok
        self.save_dir = save_dir
        self.hf_repo = hf_repo

    def on_epoch_end(self, args, state, control, **kwargs):
        ep = int(round(state.epoch or 0))
        if ep <= 0:
            return control
        ep_dir = os.path.join(self.save_dir, f"epoch-{ep}")
        self.mdl.save_pretrained(ep_dir)
        self.tok.save_pretrained(ep_dir)
        STAGE_METRICS.setdefault("sft_epochs_saved", []).append({"epoch": ep, "elapsed_min": round(elapsed_min(), 2)})
        save_metrics()
        push_folder_to_hf(ep_dir, self.hf_repo, commit_message=f"SFT epoch {ep} checkpoint")
        shutil.rmtree(ep_dir, ignore_errors=True)
        return control

def extract_loss_stats(trainer, train_result=None):
    losses = [float(x["loss"]) for x in getattr(trainer.state, "log_history", []) if isinstance(x, dict) and "loss" in x]
    tr_loss = getattr(train_result, "training_loss", None) if train_result is not None else None
    return {
        "global_step": int(getattr(trainer.state, "global_step", 0)),
        "train_loss": round(float(tr_loss), 4) if tr_loss is not None else None,
        "initial_logged_loss": round(losses[0], 4) if losses else None,
        "final_logged_loss": round(losses[-1], 4) if losses else None,
    }

def restore_adapter_in_place(mdl, adapter_dir):
    if not adapter_dir or not os.path.isdir(adapter_dir):
        print("No on-disk adapter checkpoint available to restore.")
        return False
    try:
        from peft import set_peft_model_state_dict
        st_path = os.path.join(adapter_dir, "adapter_model.safetensors")
        bin_path = os.path.join(adapter_dir, "adapter_model.bin")
        if os.path.isfile(st_path):
            from safetensors.torch import load_file
            state_dict = load_file(st_path, device="cpu")
        elif os.path.isfile(bin_path):
            state_dict = torch.load(bin_path, map_location="cpu")
        else:
            print("Adapter checkpoint missing weights file:", adapter_dir)
            return False
        set_peft_model_state_dict(mdl, state_dict)
        print("Restored verified LoRA weights in-place from:", adapter_dir)
        return True
    except Exception as e:
        print(f"Adapter restore warning: {type(e).__name__}: {str(e)[:250]}")
        return False

print("Active base model:", LOADED_BASE, "| Unsloth:", USE_UNSLOTH, "| Resumed:", bool(RESUME_FROM_ADAPTER))
model.print_trainable_parameters()

In [ ]:
# 4. Stage 0 — Continued Pre-Training (CPT) on Reference Textbooks (with OCR noise filter & SFT replay)
import collections, gc, inspect, json, math, os, random, time, torch
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

def is_clean_cpt_chunk(rec):
    """Language-aware quality gate for textbook & OCR chunks (keeps clean Spanish/English prose, drops OCR soup)."""
    raw_text = str(rec.get("text") or "")
    body = raw_text
    if "\n\n" in raw_text:
        head, tail = raw_text.split("\n\n", 1)
        if "\n" not in head and len(head) < 300:
            body = tail
    body = body.strip()
    words = body.split()
    if len(words) < 120:
        return False, "too_short"
    non_ws = [c for c in body if not c.isspace()]
    alpha_ratio = sum(c.isalpha() for c in non_ws) / max(1, len(non_ws))
    if alpha_ratio < 0.62:
        return False, "low_alpha"
    single_char_ratio = sum(len(w) == 1 and not w.isdigit() for w in words) / len(words)
    if single_char_ratio > 0.22:
        return False, "high_single_char"
    short_word_ratio = sum(len(w) <= 2 for w in words) / len(words)
    cap = 0.55 if rec.get("lang") == "es" else 0.48
    if short_word_ratio > cap:
        return False, "high_short_words"
    weird_ratio = sum(
        c == "\ufffd" or (ord(c) < 32 and c not in "\n\t\r") or (ord(c) > 0x2FFF)
        for c in body
    ) / max(1, len(body))
    if weird_ratio > 0.04:
        return False, "mojibake"
    avg_word_len = sum(len(w) for w in words) / len(words)
    if not (3.0 <= avg_word_len <= 14.0):
        return False, "bad_word_len"
    return True, "ok"

if RUN_CPT and BOOKS_FILE and os.path.exists(BOOKS_FILE) and remaining_min() > 45:
    t0 = time.time()
    assert hasattr(model, "peft_config"), "Model must already carry a LoRA adapter from Cell 3."
    if USE_UNSLOTH:
        FastLanguageModel.for_training(model)
    model.config.use_cache = False
    gc.collect()
    torch.cuda.empty_cache()

    rng = random.Random(42)
    eos = tokenizer.eos_token or ""
    raw_count = 0
    drop_reasons = collections.Counter()
    kept_by_subj_book = collections.defaultdict(lambda: collections.defaultdict(list))

    with open(BOOKS_FILE, encoding="utf-8") as fh:
        for line in fh:
            if not line.strip():
                continue
            raw_count += 1
            rec = json.loads(line)
            ok, reason = is_clean_cpt_chunk(rec)
            if not ok:
                drop_reasons[reason] += 1
                continue
            txt = str(rec["text"]).strip()
            if eos and not txt.endswith(eos):
                txt = txt + eos
            subj = str(rec.get("subject") or "unknown")
            bid = str(rec.get("book_id") or "unknown")
            kept_by_subj_book[subj][bid].append(txt)

    weighted_book_texts = []
    subj_final_counts = collections.Counter()
    for subj, by_book in sorted(kept_by_subj_book.items()):
        w = float((CPT_SUBJECT_WEIGHTS or {}).get(subj, 1.0))
        w = max(0.0, min(2.0, w))
        if w == 0.0:
            continue
        subj_pool = []
        for bid, chunks in sorted(by_book.items()):
            if w < 1.0:
                k = max(1, round(len(chunks) * w))
                sampled = rng.sample(chunks, min(k, len(chunks)))
                subj_pool.extend(sampled)
            else:
                subj_pool.extend(chunks)
        if w > 1.0 and subj_pool:
            extra = round(len(subj_pool) * (w - 1.0))
            if extra > 0:
                subj_pool.extend(rng.choices(subj_pool, k=extra))
        subj_final_counts[subj] = len(subj_pool)
        weighted_book_texts.extend(subj_pool)

    # Anti-forgetting replay from train_sft_enriched.jsonl (NEVER eval_holdout.jsonl!)
    replay_texts = []
    sft_replay_file = os.path.join(DATA_DIR, "train_sft_enriched.jsonl")
    if not os.path.exists(sft_replay_file):
        sft_replay_file = os.path.join(DATA_DIR, "train_sft.jsonl")
    if CPT_SFT_REPLAY_RATIO > 0 and os.path.exists(sft_replay_file) and weighted_book_texts:
        sft_by_subj = collections.defaultdict(list)
        with open(sft_replay_file, encoding="utf-8") as fh:
            for line in fh:
                if not line.strip():
                    continue
                r = json.loads(line)
                if r.get("messages"):
                    sft_by_subj[str(r.get("subject") or "general")].append(
                        format_chat(r["messages"], add_generation_prompt=False)
                    )
        for vals in sft_by_subj.values():
            rng.shuffle(vals)
        n_replay = max(1, int(len(weighted_book_texts) * CPT_SFT_REPLAY_RATIO / max(0.05, 1.0 - CPT_SFT_REPLAY_RATIO)))
        idx = 0
        while len(replay_texts) < n_replay:
            added = False
            for s in sorted(sft_by_subj):
                if idx < len(sft_by_subj[s]):
                    t_chat = sft_by_subj[s][idx].strip()
                    if eos and not t_chat.endswith(eos):
                        t_chat = t_chat + eos
                    replay_texts.append(t_chat)
                    added = True
                    if len(replay_texts) >= n_replay:
                        break
            if not added:
                break
            idx += 1

    cpt_texts = weighted_book_texts + replay_texts
    rng.shuffle(cpt_texts)

    # Fast token count for packing step budget (zero book text printed!)
    total_tokens = 0
    for start in range(0, len(cpt_texts), 256):
        enc = tokenizer(cpt_texts[start:start + 256], add_special_tokens=False, truncation=False)
        total_tokens += sum(len(ids) for ids in enc["input_ids"])

    print(
        f"CPT corpus summary: raw={raw_count}, dropped={dict(drop_reasons)}, "
        f"weighted_books={len(weighted_book_texts)}, sft_replay={len(replay_texts)}, "
        f"total_chunks={len(cpt_texts)}, est_tokens={total_tokens:,}"
    )
    print("CPT subject chunk distribution:", dict(subj_final_counts))

    cpt_ds = Dataset.from_list([{"text": t} for t in cpt_texts])
    cpt_out = os.path.join(OUT_DIR, f"{RECIPE['name']}-cpt")
    use_bf16 = torch.cuda.is_bf16_supported()
    cfg_params = set(inspect.signature(SFTConfig.__init__).parameters)
    tr_params = set(inspect.signature(SFTTrainer.__init__).parameters)

    seq_attempts = [(CPT_MAX_SEQ_LEN, 1, 16)]
    if CPT_FALLBACK_SEQ_LEN and CPT_FALLBACK_SEQ_LEN != CPT_MAX_SEQ_LEN:
        ga_fb = max(1, (CPT_MAX_SEQ_LEN * 16) // CPT_FALLBACK_SEQ_LEN)
        seq_attempts.append((CPT_FALLBACK_SEQ_LEN, 1, ga_fb))

    cpt_result = None
    cpt_trainer = None
    used_seq_len = CPT_MAX_SEQ_LEN
    for seq_len, bs, ga in seq_attempts:
        try:
            tok_per_step = seq_len * bs * ga
            est_steps = max(1, math.ceil(total_tokens / max(1, tok_per_step))) * CPT_EPOCHS
            max_steps_cap = CPT_MAX_STEPS if (CPT_MAX_STEPS and CPT_MAX_STEPS > 0 and est_steps > CPT_MAX_STEPS) else -1
            print(f"Starting Stage 0 CPT (seq_len={seq_len}, bs={bs}, ga={ga}, est_steps={est_steps}, max_steps={max_steps_cap}, lr={CPT_LR}) ...")

            cfg_kwargs = {
                "output_dir": cpt_out,
                "per_device_train_batch_size": bs,
                "gradient_accumulation_steps": ga,
                "num_train_epochs": CPT_EPOCHS,
                "max_steps": max_steps_cap,
                "learning_rate": CPT_LR,
                "lr_scheduler_type": "cosine",
                "warmup_ratio": 0.05,
                "fp16": not use_bf16,
                "bf16": use_bf16,
                "optim": "paged_adamw_8bit",
                "logging_steps": 10,
                "save_strategy": "steps",
                "save_steps": 50,
                "save_total_limit": 1,
                "report_to": "none",
                "seed": 42,
                "dataset_text_field": "text",
                "packing": True,
            }
            if "max_length" in cfg_params:
                cfg_kwargs["max_length"] = seq_len
            elif "max_seq_length" in cfg_params:
                cfg_kwargs["max_seq_length"] = seq_len
            cfg_kwargs = {k: v for k, v in cfg_kwargs.items() if k in cfg_params}

            tr_kwargs = {
                "model": model,
                "train_dataset": cpt_ds,
                "args": SFTConfig(**cfg_kwargs),
                "callbacks": [TimeBudgetCallback(reserve_min=45.0, label="CPT")],
            }
            if "processing_class" in tr_params:
                tr_kwargs["processing_class"] = tokenizer
            else:
                tr_kwargs["tokenizer"] = tokenizer

            cpt_trainer = SFTTrainer(**tr_kwargs)
            cpt_result = cpt_trainer.train()
            used_seq_len = seq_len
            break
        except (torch.cuda.OutOfMemoryError, RuntimeError) as oom_err:
            if "out of memory" in str(oom_err).lower() and seq_len != seq_attempts[-1][0]:
                print(f"OOM at CPT seq_len={seq_len}; freeing VRAM and retrying with seq_len={seq_attempts[-1][0]} ...")
                gc.collect()
                torch.cuda.empty_cache()
                continue
            raise

    model.save_pretrained(cpt_out)
    tokenizer.save_pretrained(cpt_out)
    cleanup_trainer_checkpoints(cpt_out)
    active_adapter = cpt_out

    stats = extract_loss_stats(cpt_trainer, cpt_result)
    stats.update({
        "status": "completed",
        "minutes": round((time.time() - t0) / 60.0, 2),
        "seq_len": used_seq_len,
        "raw_chunks": raw_count,
        "kept_book_chunks": len(weighted_book_texts),
        "sft_replay_rows": len(replay_texts),
        "est_tokens": total_tokens,
    })
    hf_repo = push_folder_to_hf(cpt_out, f"{HF_USERNAME}/{HF_REPO_BASE}-cpt", commit_message="Stage 0 CPT adapter")
    stats["hf_repo"] = hf_repo
    STAGE_METRICS["cpt"] = stats
    save_metrics()
    print("Stage 0 CPT complete!", stats)
else:
    reason = "disabled" if not RUN_CPT else ("books_file_missing" if not BOOKS_FILE else "insufficient_time_budget")
    STAGE_METRICS["cpt"] = {"status": f"skipped ({reason})"}
    save_metrics()
    print(f"Skipping Stage 0 CPT ({reason}).")

In [ ]:
# 5. Stage 1 — Supervised Fine-Tuning (SFT on train_sft_enriched.jsonl, 5,926 examples)
import gc, inspect, json, os, time, torch
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

if RUN_SFT and remaining_min() > 45:
    t0 = time.time()
    assert hasattr(model, "peft_config"), "Model must already carry a LoRA adapter (do not recreate LoRA after CPT)."
    if USE_UNSLOTH:
        FastLanguageModel.for_training(model)
    model.config.use_cache = False
    gc.collect()
    torch.cuda.empty_cache()

    sft_file = os.path.join(DATA_DIR, "train_sft_enriched.jsonl")
    if not os.path.exists(sft_file):
        sft_file = os.path.join(DATA_DIR, "train_sft.jsonl")
    print("Using SFT dataset:", sft_file)

    with open(sft_file, encoding="utf-8") as f:
        raw_rows = [json.loads(line) for line in f if line.strip()]
    sft_ds = Dataset.from_list([
        {"text": format_chat(r["messages"], add_generation_prompt=False)}
        for r in raw_rows if r.get("messages")
    ])
    print(f"Prepared {len(sft_ds)} formatted SFT examples (reusing existing LoRA: {bool(active_adapter)}).")

    sft_out = os.path.join(OUT_DIR, f"{RECIPE['name']}-sft")
    use_bf16 = torch.cuda.is_bf16_supported()

    cfg_params = set(inspect.signature(SFTConfig.__init__).parameters)
    cfg_kwargs = {
        "output_dir": sft_out,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 16,
        "num_train_epochs": SFT_EPOCHS,
        "max_steps": SFT_MAX_STEPS if (SFT_MAX_STEPS and SFT_MAX_STEPS > 0) else -1,
        "learning_rate": SFT_LR,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.05,
        "fp16": not use_bf16,
        "bf16": use_bf16,
        "optim": "paged_adamw_8bit",
        "logging_steps": 10,
        "save_strategy": "epoch",
        "save_total_limit": 1,
        "report_to": "none",
        "seed": 42,
        "dataset_text_field": "text",
        "packing": False,
    }
    if "max_length" in cfg_params:
        cfg_kwargs["max_length"] = MAX_SEQ_LEN
    elif "max_seq_length" in cfg_params:
        cfg_kwargs["max_seq_length"] = MAX_SEQ_LEN
    cfg_kwargs = {k: v for k, v in cfg_kwargs.items() if k in cfg_params}

    sft_hf_repo = f"{HF_USERNAME}/{HF_REPO_BASE}-sft"
    tr_params = set(inspect.signature(SFTTrainer.__init__).parameters)
    tr_kwargs = {
        "model": model,
        "train_dataset": sft_ds,
        "args": SFTConfig(**cfg_kwargs),
        "callbacks": [
            TimeBudgetCallback(reserve_min=45.0 + PLANNED_MIN_AFTER_SFT, label="SFT"),
            EpochPushCallback(model, tokenizer, sft_out, sft_hf_repo),
        ],
    }
    if "processing_class" in tr_params:
        tr_kwargs["processing_class"] = tokenizer
    else:
        tr_kwargs["tokenizer"] = tokenizer

    trainer = SFTTrainer(**tr_kwargs)
    sft_res = trainer.train()

    model.save_pretrained(sft_out)
    tokenizer.save_pretrained(sft_out)
    cleanup_trainer_checkpoints(sft_out)
    active_adapter = sft_out

    stats = extract_loss_stats(trainer, sft_res)
    stats.update({
        "status": "completed",
        "minutes": round((time.time() - t0) / 60.0, 2),
        "examples": len(sft_ds),
        "epochs": SFT_EPOCHS,
    })
    hf_repo = push_folder_to_hf(sft_out, sft_hf_repo, commit_message=f"Stage 1 SFT final ({SFT_EPOCHS} epochs)")
    stats["hf_repo"] = hf_repo
    STAGE_METRICS["sft"] = stats
    save_metrics()
    print("Stage 1 SFT complete!", stats)
else:
    reason = "disabled" if not RUN_SFT else "insufficient_time_budget"
    STAGE_METRICS["sft"] = {"status": f"skipped ({reason})"}
    save_metrics()
    print(f"Skipping Stage 1 SFT ({reason}).")

In [ ]:
# 6. Stage 2 — DPO Preference Alignment (1,310 pairs, fault-tolerant checkpointing)
import gc, inspect, json, os, time, torch
from datasets import Dataset

dpo_file = os.path.join(DATA_DIR, "train_dpo_enriched.jsonl")
if not os.path.exists(dpo_file):
    dpo_file = os.path.join(DATA_DIR, "train_dpo.jsonl")

if RUN_DPO and os.path.exists(dpo_file) and remaining_min() > 45:
    t0 = time.time()
    try:
        if USE_UNSLOTH:
            FastLanguageModel.for_training(model)
            try:
                from unsloth import PatchDPOTrainer
                PatchDPOTrainer()
            except Exception as e:
                print("PatchDPOTrainer skipped:", e)
        model.config.use_cache = False
        gc.collect()
        torch.cuda.empty_cache()

        from trl import DPOConfig, DPOTrainer

        def msg_text(val):
            if isinstance(val, str):
                return val
            if isinstance(val, list) and val:
                return str(val[-1].get("content", "")) if isinstance(val[-1], dict) else str(val[-1])
            return str(val or "")

        with open(dpo_file, encoding="utf-8") as f:
            raw_dpo = [json.loads(line) for line in f if line.strip()]
        dpo_ds = Dataset.from_list([
            {
                "prompt": format_chat(r["prompt"], add_generation_prompt=True) if isinstance(r["prompt"], list) else str(r["prompt"]),
                "chosen": msg_text(r["chosen"]),
                "rejected": msg_text(r["rejected"]),
            }
            for r in raw_dpo
        ])
        print(f"Prepared {len(dpo_ds)} DPO preference pairs from {dpo_file}.")

        dpo_out = os.path.join(OUT_DIR, f"{RECIPE['name']}-dpo")
        use_bf16 = torch.cuda.is_bf16_supported()
        dpo_cfg_params = set(inspect.signature(DPOConfig.__init__).parameters)
        dpo_cfg_kwargs = {
            "output_dir": dpo_out,
            "per_device_train_batch_size": 1,
            "gradient_accumulation_steps": 8,
            "num_train_epochs": 1,
            "max_steps": DPO_MAX_STEPS if (DPO_MAX_STEPS and DPO_MAX_STEPS > 0) else -1,
            "learning_rate": DPO_LR,
            "beta": DPO_BETA,
            "fp16": not use_bf16,
            "bf16": use_bf16,
            "optim": "paged_adamw_8bit",
            "logging_steps": 10,
            "save_strategy": "no",
            "report_to": "none",
            "max_length": MAX_SEQ_LEN,
            "max_prompt_length": 1024,
            "seed": 42,
        }
        dpo_cfg_kwargs = {k: v for k, v in dpo_cfg_kwargs.items() if k in dpo_cfg_params}

        dpo_tr_params = set(inspect.signature(DPOTrainer.__init__).parameters)
        dpo_tr_kwargs = {
            "model": model,
            "ref_model": None,
            "train_dataset": dpo_ds,
            "args": DPOConfig(**dpo_cfg_kwargs),
            "callbacks": [TimeBudgetCallback(reserve_min=40.0 + PLANNED_MIN_AFTER_DPO, label="DPO")],
        }
        if "processing_class" in dpo_tr_params:
            dpo_tr_kwargs["processing_class"] = tokenizer
        else:
            dpo_tr_kwargs["tokenizer"] = tokenizer

        dpo_trainer = DPOTrainer(**dpo_tr_kwargs)
        dpo_res = dpo_trainer.train()

        model.save_pretrained(dpo_out)
        tokenizer.save_pretrained(dpo_out)
        cleanup_trainer_checkpoints(dpo_out)
        active_adapter = dpo_out

        stats = extract_loss_stats(dpo_trainer, dpo_res)
        stats.update({"status": "completed", "minutes": round((time.time() - t0) / 60.0, 2), "pairs": len(dpo_ds)})
        hf_repo = push_folder_to_hf(dpo_out, f"{HF_USERNAME}/{HF_REPO_BASE}-dpo", commit_message="Stage 2 DPO adapter")
        stats["hf_repo"] = hf_repo
        STAGE_METRICS["dpo"] = stats
        save_metrics()
        print("Stage 2 DPO complete!", stats)
    except Exception as e:
        print(f"DPO warning (preserving last good adapter at {active_adapter}):", type(e).__name__, str(e)[:400])
        STAGE_METRICS["dpo"] = {"status": "failed", "error": f"{type(e).__name__}: {str(e)[:200]}", "minutes": round((time.time() - t0) / 60.0, 2)}
        save_metrics()
        gc.collect()
        torch.cuda.empty_cache()
        restore_adapter_in_place(model, active_adapter)
else:
    reason = "disabled" if not RUN_DPO else ("file_missing" if not os.path.exists(dpo_file) else "insufficient_time_budget")
    STAGE_METRICS["dpo"] = {"status": f"skipped ({reason})"}
    save_metrics()
    print(f"Skipping Stage 2 DPO ({reason}).")

In [ ]:
# 7. Stage 3 — GRPO Verifiable RL (KaTeX + No-Skip Rigour + Socratic Guard + SymPy)
import gc, inspect, json, os, sys, time, torch
from datasets import Dataset

grpo_file = os.path.join(DATA_DIR, "train_grpo.jsonl")
rewards_py = os.path.join(DATA_DIR, "rewards.py")

if RUN_GRPO and os.path.exists(grpo_file) and os.path.exists(rewards_py) and remaining_min() > 45:
    t0 = time.time()
    try:
        if USE_UNSLOTH:
            FastLanguageModel.for_training(model)
        model.config.use_cache = False
        gc.collect()
        torch.cuda.empty_cache()

        if DATA_DIR not in sys.path:
            sys.path.insert(0, DATA_DIR)
        from rewards import katex_syntax_reward, no_skip_rigour_reward, socratic_guard_reward, sympy_answer_reward
        from trl import GRPOConfig, GRPOTrainer

        def boxed_final_reward(completions, expected_answer=None, socratic=None, **_kwargs):
            """+0.3 when a non-Socratic prompt with a gold answer ends in \\boxed{...}, -0.2 when it does not."""
            n = len(completions)
            golds = expected_answer if isinstance(expected_answer, list) else [expected_answer] * n
            socs = socratic if isinstance(socratic, list) else [bool(socratic)] * n
            out = []
            for c, g, soc in zip(completions, golds, socs):
                text = c if isinstance(c, str) else (c[-1].get("content", "") if isinstance(c, list) and c and isinstance(c[-1], dict) else str(c))
                if soc or not g:
                    out.append(0.0)
                else:
                    out.append(0.3 if "\\boxed{" in text else -0.2)
            return out

        grpo_rewards = [katex_syntax_reward, no_skip_rigour_reward, socratic_guard_reward, sympy_answer_reward]
        if GRPO_BOXED_REWARD:
            grpo_rewards.append(boxed_final_reward)

        with open(grpo_file, encoding="utf-8") as f:
            raw_grpo = [json.loads(line) for line in f if line.strip()]
        grpo_ds = Dataset.from_list([
            {
                "prompt": format_chat(r["prompt"], add_generation_prompt=True) if isinstance(r["prompt"], list) else str(r["prompt"]),
                "socratic": bool(r.get("socratic", False)),
                "expected_answer": str(r.get("expected_answer") or ""),
                "forbidden_answers": str(r.get("forbidden_answers") or ""),
            }
            for r in raw_grpo
        ])
        print(f"Loaded {len(grpo_ds)} GRPO prompts (running up to {GRPO_MAX_STEPS} steps).")

        grpo_out = os.path.join(OUT_DIR, f"{RECIPE['name']}-grpo")
        use_bf16 = torch.cuda.is_bf16_supported()
        g_params = set(inspect.signature(GRPOConfig.__init__).parameters)
        g_kwargs = {
            "output_dir": grpo_out,
            "learning_rate": 5e-6,
            "per_device_train_batch_size": 1,
            "gradient_accumulation_steps": 4,
            "num_generations": 4,
            "max_prompt_length": 1024,
            "max_completion_length": 512,
            "max_steps": GRPO_MAX_STEPS,
            "fp16": not use_bf16,
            "bf16": use_bf16,
            "optim": "paged_adamw_8bit",
            "logging_steps": 5,
            "save_strategy": "no",
            "report_to": "none",
            "seed": 42,
        }
        g_kwargs = {k: v for k, v in g_kwargs.items() if k in g_params}

        gt_params = set(inspect.signature(GRPOTrainer.__init__).parameters)
        gt_kwargs = {
            "model": model,
            "reward_funcs": grpo_rewards,
            "train_dataset": grpo_ds,
            "args": GRPOConfig(**g_kwargs),
            "callbacks": [TimeBudgetCallback(reserve_min=20.0 + PLANNED_MIN_EVAL_EXPORT, label="GRPO")],
        }
        if "processing_class" in gt_params:
            gt_kwargs["processing_class"] = tokenizer
        else:
            gt_kwargs["tokenizer"] = tokenizer

        grpo_trainer = GRPOTrainer(**gt_kwargs)
        grpo_res = grpo_trainer.train()

        model.save_pretrained(grpo_out)
        tokenizer.save_pretrained(grpo_out)
        cleanup_trainer_checkpoints(grpo_out)
        active_adapter = grpo_out

        stats = extract_loss_stats(grpo_trainer, grpo_res)
        stats.update({"status": "completed", "minutes": round((time.time() - t0) / 60.0, 2), "max_steps": GRPO_MAX_STEPS})
        hf_repo = push_folder_to_hf(grpo_out, f"{HF_USERNAME}/{HF_REPO_BASE}-grpo", commit_message="Stage 3 GRPO adapter")
        stats["hf_repo"] = hf_repo
        STAGE_METRICS["grpo"] = stats
        save_metrics()
        print("Stage 3 GRPO complete!", stats)
    except Exception as e:
        print(f"GRPO warning (preserving last good adapter at {active_adapter}):", type(e).__name__, str(e)[:400])
        STAGE_METRICS["grpo"] = {"status": "failed", "error": f"{type(e).__name__}: {str(e)[:200]}", "minutes": round((time.time() - t0) / 60.0, 2)}
        save_metrics()
        gc.collect()
        torch.cuda.empty_cache()
        restore_adapter_in_place(model, active_adapter)
else:
    reason = "disabled" if not RUN_GRPO else ("files_missing" if not (os.path.exists(grpo_file) and os.path.exists(rewards_py)) else "insufficient_time_budget")
    STAGE_METRICS["grpo"] = {"status": f"skipped ({reason})"}
    save_metrics()
    print(f"Skipping Stage 3 GRPO ({reason}).")

In [ ]:
# 8. Quantitative & Qualitative Evaluation on eval_holdout.jsonl (KaTeX + rewards.py + Optional Base Comparison)
import collections, json, os, re, sys, time, torch

eval_file = os.path.join(DATA_DIR, "eval_holdout.jsonl")
PHASE_RULE_RE = re.compile(r"\b(?:Phase|Fase)\s*[1-4]\b", re.I)
EMOJI_RULE_RE = re.compile("[\U0001f000-\U0001faff\u2600-\u27bf\u2b00-\u2bff]")

if RUN_EVAL and os.path.exists(eval_file) and remaining_min() > 15:
    t0 = time.time()
    if DATA_DIR not in sys.path:
        sys.path.insert(0, DATA_DIR)
    from rewards import katex_syntax_reward, no_skip_rigour_reward, socratic_guard_reward, sympy_answer_reward

    with open(eval_file, encoding="utf-8") as fh:
        all_eval = [json.loads(line) for line in fh if line.strip()]

    # Deterministic stratified sampling across (subject, task)
    buckets = collections.defaultdict(list)
    for row in all_eval:
        buckets[(row.get("subject", "?"), row.get("task", "?"))].append(row)
    samples = []
    idx = 0
    while len(samples) < min(EVAL_MAX_SAMPLES, len(all_eval)):
        added = False
        for key in sorted(buckets):
            if idx < len(buckets[key]):
                samples.append(buckets[key][idx])
                added = True
                if len(samples) >= EVAL_MAX_SAMPLES:
                    break
        if not added:
            break
        idx += 1

    if USE_UNSLOTH:
        FastLanguageModel.for_inference(model)
    else:
        model.eval()
        model.config.use_cache = True

    def evaluate_model_on_samples(eval_rows, disable_adapter=False):
        katex_scores, rigour_scores, soc_scores, sympy_scores = [], [], [], []
        phase_hits, emoji_hits = 0, 0
        previews = []
        ctx_mgr = model.disable_adapter() if (disable_adapter and hasattr(model, "disable_adapter")) else None

        def _run_loop():
            nonlocal phase_hits, emoji_hits
            for s in eval_rows:
                prompt = format_chat(s["messages"][:-1], add_generation_prompt=True)
                inputs = tokenizer([prompt], return_tensors="pt", truncation=True, max_length=1536).to("cuda")
                with torch.no_grad():
                    out = model.generate(
                        **inputs,
                        max_new_tokens=EVAL_MAX_NEW_TOKENS,
                        temperature=0.2,
                        do_sample=True,
                        pad_token_id=tokenizer.pad_token_id,
                    )
                reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
                gold = str(s["messages"][-1].get("content") or "")
                is_soc = str(s.get("task", "")).startswith("socratic")
                hidden_src = str(s.get("hidden_solution") or "") + "\n" + str(s["messages"][0].get("content") or "")
                boxed_gold = re.findall(r"\\boxed\{([^{}]+(?:\{[^{}]*\}[^{}]*)*)\}", hidden_src if is_soc else gold)
                gold_expr = boxed_gold[-1].strip() if boxed_gold else ""

                k_sc = katex_syntax_reward([reply])[0]
                r_sc = no_skip_rigour_reward([reply], socratic=[is_soc])[0]
                katex_scores.append(k_sc)
                rigour_scores.append(r_sc)
                if is_soc:
                    s_sc = socratic_guard_reward([reply], socratic=[True], forbidden_answers=[gold_expr])[0]
                    soc_scores.append(s_sc)
                if (not is_soc) and gold_expr:
                    y_sc = sympy_answer_reward([reply], expected_answer=[gold_expr])[0]
                    sympy_scores.append(y_sc)
                if PHASE_RULE_RE.search(reply):
                    phase_hits += 1
                if EMOJI_RULE_RE.search(reply):
                    emoji_hits += 1
                if len(previews) < 3:
                    previews.append({
                        "id": s.get("id"),
                        "subject": s.get("subject"),
                        "task": s.get("task"),
                        "katex": k_sc,
                        "rigour": r_sc,
                        "preview": reply[:400],
                    })

        if ctx_mgr is not None:
            with ctx_mgr:
                _run_loop()
        else:
            _run_loop()

        n = max(1, len(eval_rows))
        return {
            "n": len(eval_rows),
            "katex_pass_rate": round(sum(1 for v in katex_scores if v >= 0) / n, 3),
            "katex_mean": round(sum(katex_scores) / n, 3),
            "no_skip_mean": round(sum(rigour_scores) / n, 3),
            "socratic_n": len(soc_scores),
            "socratic_mean": round(sum(soc_scores) / len(soc_scores), 3) if soc_scores else None,
            "sympy_n": len(sympy_scores),
            "sympy_acc": round(sum(1 for v in sympy_scores if v >= 0.99) / len(sympy_scores), 3) if sympy_scores else None,
            "phase_violation_rate": round(phase_hits / n, 3),
            "emoji_violation_rate": round(emoji_hits / n, 3),
            "previews": previews,
        }

    print(f"Evaluating tuned model on {len(samples)} stratified eval_holdout prompts ...")
    tuned_eval = evaluate_model_on_samples(samples, disable_adapter=False)
    eval_report = {"tuned": tuned_eval, "base": None}

    if EVAL_COMPARE_BASE and hasattr(model, "disable_adapter") and remaining_min() > 25:
        try:
            print(f"Evaluating base model ({LOADED_BASE}) on the same {len(samples)} holdout prompts ...")
            eval_report["base"] = evaluate_model_on_samples(samples, disable_adapter=True)
        except Exception as e_base:
            print("Base comparison skipped:", type(e_base).__name__, str(e_base)[:250])

    print("\n--- Holdout Evaluation Summary ---")
    for label_k in ("base", "tuned"):
        res = eval_report.get(label_k)
        if res:
            print(
                f"  [{label_k.upper():5s}] n={res['n']} | KaTeX pass={res['katex_pass_rate']:.1%} (mean {res['katex_mean']:+.2f}) | "
                f"No-Skip mean={res['no_skip_mean']:+.2f} | Socratic={res['socratic_mean']} (n={res['socratic_n']}) | "
                f"SymPy acc={res['sympy_acc']} (n={res['sympy_n']}) | Phase viol={res['phase_violation_rate']:.1%} | Emoji viol={res['emoji_violation_rate']:.1%}"
            )

    print("\n--- Qualitative Samples (eval_holdout.jsonl) ---")
    for pv in tuned_eval["previews"]:
        print(f"=== {pv['id']} ({pv['subject']} | {pv['task']}) | KaTeX={pv['katex']:+.2f} | Rigour={pv['rigour']:+.2f} ===")
        print(pv["preview"], "\n")

    eval_path = os.path.join(OUT_DIR, "eval_report.json")
    with open(eval_path, "w", encoding="utf-8") as f:
        json.dump(eval_report, f, indent=2)

    STAGE_METRICS["eval"] = {
        "status": "completed",
        "minutes": round((time.time() - t0) / 60.0, 2),
        "tuned": {k: v for k, v in tuned_eval.items() if k != "previews"},
        "base": ({k: v for k, v in eval_report["base"].items() if k != "previews"} if eval_report["base"] else None),
    }
    save_metrics()
else:
    reason = "disabled" if not RUN_EVAL else ("eval_file_missing" if not os.path.exists(eval_file) else "insufficient_time_budget")
    STAGE_METRICS["eval"] = {"status": f"skipped ({reason})"}
    save_metrics()
    print(f"Skipping Cell 8 Evaluation ({reason}).")

In [ ]:
# 9. Final Export (LoRA + Merged 16-bit + GGUF Q4_K_M) & Private HF Push + Session Summary
import gc, os, shutil, time, torch

final_lora = os.path.join(OUT_DIR, f"{RECIPE['name']}-final-lora")
merged_dir = os.path.join(OUT_DIR, f"{RECIPE['name']}-merged-16bit")
gguf_dir   = os.path.join(OUT_DIR, f"{RECIPE['name']}-gguf")

try:
    model.save_pretrained(final_lora)
    tokenizer.save_pretrained(final_lora)
    print("Saved final LoRA adapter ->", final_lora)
except Exception as e_save:
    print(f"Direct save_pretrained failed ({type(e_save).__name__}); using last verified adapter at {active_adapter}")
    final_lora = active_adapter if (active_adapter and os.path.isdir(active_adapter)) else final_lora

metrics_path = save_metrics()
eval_json = os.path.join(OUT_DIR, "eval_report.json")
if os.path.isdir(final_lora):
    shutil.copy2(metrics_path, os.path.join(final_lora, "training_metrics.json"))
    if os.path.isfile(eval_json):
        shutil.copy2(eval_json, os.path.join(final_lora, "eval_report.json"))
    push_folder_to_hf(final_lora, f"{HF_USERNAME}/{HF_REPO_BASE}-lora", commit_message="Final LoRA adapter & metrics")

if RUN_EXPORT:
    t0 = time.time()
    merged_ok = False
    merged_pushed = False
    if USE_UNSLOTH:
        try:
            model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")
            merged_ok = True
            print("Saved merged 16-bit model ->", merged_dir)
        except Exception as e:
            print("Merged 16-bit export skipped:", type(e).__name__, str(e)[:300])
    else:
        try:
            merged = model.merge_and_unload()
            merged.save_pretrained(merged_dir, safe_serialization=True)
            tokenizer.save_pretrained(merged_dir)
            merged_ok = True
            print("Saved merged 16-bit model ->", merged_dir)
        except Exception as e:
            print("Merge skipped (final LoRA adapter is safe):", type(e).__name__, str(e)[:300])

    if merged_ok and os.path.isdir(merged_dir):
        shutil.copy2(metrics_path, os.path.join(merged_dir, "training_metrics.json"))
        if os.path.isfile(eval_json):
            shutil.copy2(eval_json, os.path.join(merged_dir, "eval_report.json"))
        merged_pushed = bool(push_folder_to_hf(merged_dir, f"{HF_USERNAME}/{HF_REPO_BASE}", commit_message="Merged 16-bit Sideris model"))

    # Guard against Kaggle's 20 GB /kaggle/working disk limit before writing GGUF
    free_gb = shutil.disk_usage(OUT_DIR).free / (1024 ** 3)
    if merged_pushed and free_gb < 11.0 and os.path.isdir(merged_dir):
        print(f"Disk guard: {free_gb:.1f} GB free after HF upload; clearing merged_16bit before GGUF export ...")
        shutil.rmtree(merged_dir, ignore_errors=True)
        gc.collect()

    gguf_ok = False
    if USE_UNSLOTH:
        try:
            model.save_pretrained_gguf(gguf_dir, tokenizer, quantization_method="q4_k_m")
            gguf_ok = True
            print("Saved GGUF Q4_K_M ->", gguf_dir)
            push_folder_to_hf(gguf_dir, f"{HF_USERNAME}/{HF_REPO_BASE}-GGUF", commit_message="GGUF Q4_K_M Sideris model")
        except Exception as e:
            print("GGUF export skipped (LoRA & merged 16-bit are safe):", type(e).__name__, str(e)[:300])

    STAGE_METRICS["export"] = {
        "status": "completed",
        "minutes": round((time.time() - t0) / 60.0, 2),
        "merged_16bit": merged_ok,
        "gguf_q4_k_m": gguf_ok,
    }
    save_metrics()
else:
    STAGE_METRICS["export"] = {"status": "skipped (RUN_EXPORT=False; checkpoints already pushed to HF)"}
    save_metrics()
    print("Full 16-bit/GGUF export skipped (RUN_EXPORT=False). Private LoRA checkpoints are on HF.")

print("\n================ SIDERIS SESSION SUMMARY ================")
print(f"Model: {RECIPE['name']} ({LOADED_BASE}) | Total elapsed: {elapsed_min():.1f} min")
for stage_name in ("cpt", "sft", "dpo", "grpo", "eval", "export"):
    st = STAGE_METRICS.get(stage_name, {})
    print(
        f"  {stage_name.upper():6s} | status={st.get('status', 'not_run')} | "
        f"min={st.get('minutes', '-')} | step={st.get('global_step', '-')} | "
        f"loss={st.get('final_logged_loss', st.get('train_loss', '-'))} | hf={st.get('hf_repo', '-')}"
    )
print("=========================================================")